# Any-wasting prediction with XGBoost and LightGBM

This notebook fits **XGBoost and LightGBM only**. Optuna and CatBoost have been removed. Both models use the same 23 predictors, SMOTENC within training folds, the MICS child sampling weight (`ch_weight`, falling back to `chweight`), and a model-specific classification threshold selected by maximizing survey-weighted F1 on training-only out-of-fold predictions.

**Weight caveat:** synthetic SMOTENC rows have no original MICS survey weight. Each synthetic row is assigned the mean minority-class weight from its training fold; original rows retain their own weights. This is an approximation, not a survey-design estimator. Run a no-SMOTE sensitivity analysis before drawing substantive conclusions.

## 1. Install dependencies

In [ ]:
import importlib.util, subprocess, sys
packages={"numpy":"numpy","pandas":"pandas","sklearn":"scikit-learn",
          "imblearn":"imbalanced-learn","xgboost":"xgboost","lightgbm":"lightgbm","matplotlib":"matplotlib"}
missing=[pip for mod,pip in packages.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.check_call([sys.executable,"-m","pip","install",*missing])
print("Dependencies ready.")

## 2. Imports and reproducibility

In [ ]:
import json, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from imblearn.over_sampling import SMOTENC
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import (roc_auc_score,average_precision_score,accuracy_score,
    precision_score,recall_score,f1_score,confusion_matrix,classification_report,
    precision_recall_curve,RocCurveDisplay,PrecisionRecallDisplay)
warnings.filterwarnings("ignore")
SEED=42
CV_FOLDS=5
np.random.seed(SEED)

## 3. Load the revised MICS child dataset

In [ ]:
DATA_PATH=Path("MICS-child-ISRC-revised.dta")
if not DATA_PATH.exists():
    candidates=list(dict.fromkeys(p for p in [Path("MICS-child-ISRC.dta"),*Path(".").glob("*.dta")] if p.exists()))
    if len(candidates)!=1:
        raise FileNotFoundError(f"Set DATA_PATH explicitly; found: {candidates}")
    DATA_PATH=candidates[0]
df=pd.read_stata(DATA_PATH,convert_categoricals=False)
print("Loaded:",DATA_PATH,"| shape:",df.shape)
print("Weight-like fields:",[c for c in df.columns if "weight" in c.lower()])

## 4. Define outcome, predictors and child sample weight

In [ ]:
TARGET="any_wasting"
PREDICTORS=["age_m","female","division","urban","wb4","parity_cat","diarrhoea_2w",
"cough_2w","breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level","electricity",
"wealth","birth_order","hh_size","water_treated","water_boil_filter","handwash_place",
"handwash_water","floor_quality","ari_chest"]
WEIGHT_CANDIDATES=["ch_weight","chweight"]
WEIGHT_COL=next((c for c in WEIGHT_CANDIDATES if c in df.columns),None)
if WEIGHT_COL is None:
    raise KeyError("Child sample weight not found. Expected ch_weight or chweight; verify the Stata variable name.")
required=[TARGET]+PREDICTORS+[WEIGHT_COL]
missing=[c for c in required if c not in df.columns]
if missing: raise KeyError(f"Missing required columns: {missing}")
model_df=df[required].copy()
model_df[WEIGHT_COL]=pd.to_numeric(model_df[WEIGHT_COL],errors="coerce")
model_df=model_df.dropna(subset=required)
model_df=model_df.loc[model_df[WEIGHT_COL]>0].copy()
model_df[TARGET]=pd.to_numeric(model_df[TARGET],errors="raise").astype(int)
if sorted(model_df[TARGET].unique())!=[0,1]: raise ValueError("any_wasting must be coded 0/1.")
print("Weight column:",WEIGHT_COL,"| complete cases:",len(model_df))
print("Outcome counts:",model_df[TARGET].value_counts().sort_index().to_dict())
display(model_df[WEIGHT_COL].describe().to_frame("child sample weight"))

## 5. Feature types and stratified holdout split

In [ ]:
CATEGORICAL_COLUMNS=["female","division","urban","parity_cat","diarrhoea_2w","cough_2w",
"breathing_2w","fuel_cat","water_cat","toilet_cat","edu_level","electricity","wealth",
"birth_order","water_treated","water_boil_filter","handwash_place","handwash_water",
"floor_quality","ari_chest"]
NUMERIC_COLUMNS=[c for c in PREDICTORS if c not in CATEGORICAL_COLUMNS]
X=model_df[PREDICTORS].copy()
for c in CATEGORICAL_COLUMNS: X[c]=X[c].astype(str)
for c in NUMERIC_COLUMNS: X[c]=pd.to_numeric(X[c],errors="raise")
y=model_df[TARGET].astype(int)
w=model_df[WEIGHT_COL].astype(float)
X_train,X_test,y_train,y_test,w_train,w_test=train_test_split(
    X,y,w,test_size=.20,random_state=SEED,stratify=y)
print("Train:",len(X_train),"positive:",int(y_train.sum()))
print("Test:",len(X_test),"positive:",int(y_test.sum()))

## 6. SMOTENC and sample-weight handling

SMOTENC is applied only to each training fold. Validation and held-out test data remain untouched. Original rows keep their child sampling weights; synthetic rows are assigned the mean minority-class weight from the current training fold.

In [ ]:
CAT_INDICES=[X.columns.get_loc(c) for c in CATEGORICAL_COLUMNS]
def smotenc_resample(Xf,yf,wf):
    counts=pd.Series(yf).value_counts()
    if counts.min()<2: raise ValueError("Not enough minority observations for SMOTENC.")
    sampler=SMOTENC(categorical_features=CAT_INDICES,random_state=SEED,
                    k_neighbors=min(5,int(counts.min())-1))
    Xr,yr=sampler.fit_resample(Xf,yf)
    if not isinstance(Xr,pd.DataFrame): Xr=pd.DataFrame(Xr,columns=Xf.columns)
    for col in CATEGORICAL_COLUMNS: Xr[col]=Xr[col].astype(str)
    for col in NUMERIC_COLUMNS: Xr[col]=pd.to_numeric(Xr[col],errors="raise")
    n_synthetic=len(yr)-len(yf)
    minority=counts.idxmin()
    wf=np.asarray(wf,dtype=float)
    synthetic_weight=wf[np.asarray(yf)==minority].mean()
    wr=np.r_[wf,np.repeat(synthetic_weight,n_synthetic)]
    return Xr,np.asarray(yr,dtype=int),wr/wr.mean()

def make_preprocessor():
    return ColumnTransformer([
        ("categorical",OneHotEncoder(handle_unknown="ignore"),CATEGORICAL_COLUMNS),
        ("numeric","passthrough",NUMERIC_COLUMNS)
    ])

# Fixed, documented configurations: no hyperparameter search is performed.
MODEL_PARAMS={
 "XGBoost":{"n_estimators":700,"max_depth":5,"learning_rate":0.05,
            "min_child_weight":3,"subsample":0.85,"colsample_bytree":0.85,
            "reg_lambda":5.0},
 "LightGBM":{"n_estimators":700,"num_leaves":31,"max_depth":-1,
             "learning_rate":0.05,"min_child_samples":25,
             "subsample":0.85,"subsample_freq":1,"colsample_bytree":0.85,
             "reg_lambda":5.0}
}
def fit_model(name,Xf,yf,wf):
    Xr,yr,wr=smotenc_resample(Xf,yf,wf)
    prep=make_preprocessor()
    Xr_encoded=prep.fit_transform(Xr)
    if name=="XGBoost":
        model=XGBClassifier(objective="binary:logistic",eval_metric="logloss",
            tree_method="hist",n_jobs=1,random_state=SEED,verbosity=0,**MODEL_PARAMS[name])
    elif name=="LightGBM":
        model=LGBMClassifier(objective="binary",n_jobs=1,random_state=SEED,
            verbosity=-1,**MODEL_PARAMS[name])
    else: raise ValueError(name)
    model.fit(Xr_encoded,yr,sample_weight=wr)
    return {"model":model,"preprocessor":prep}

def predict_probability(bundle,Xe):
    return bundle["model"].predict_proba(bundle["preprocessor"].transform(Xe))[:,1]
print("Models:",list(MODEL_PARAMS))
print("SMOTENC categorical predictors:",len(CAT_INDICES))

## 7. Five-fold cross-validation using weighted PR-AUC

The fixed model configurations are assessed using stratified five-fold CV. The reported score is the mean survey-weighted average precision across validation folds. No Optuna or other hyperparameter optimization is performed.

In [ ]:
cv=StratifiedKFold(n_splits=CV_FOLDS,shuffle=True,random_state=SEED)
def cross_validated_score(name):
    scores=[]
    for fold,(tr,va) in enumerate(cv.split(X_train,y_train),start=1):
        bundle=fit_model(name,X_train.iloc[tr],y_train.iloc[tr],w_train.iloc[tr])
        prob=predict_probability(bundle,X_train.iloc[va])
        score=average_precision_score(y_train.iloc[va],prob,sample_weight=w_train.iloc[va])
        scores.append(float(score))
        print(f"{name} fold {fold}: weighted PR-AUC={score:.5f}")
    return float(np.mean(scores)),scores

cv_rows=[]
for name in MODEL_PARAMS:
    mean_score,fold_scores=cross_validated_score(name)
    cv_rows.append({"model":name,"mean_weighted_cv_pr_auc":mean_score,
                    "fold_scores":fold_scores})
cv_results=pd.DataFrame(cv_rows)
display(cv_results)

## 8. Optimize classification threshold from training-only out-of-fold predictions

For each model, collect out-of-fold probabilities on the training partition. Select the threshold maximizing survey-weighted F1. The held-out test partition is not used for this selection.

In [ ]:
def get_oof_probabilities(name):
    oof=np.full(len(X_train),np.nan)
    for tr,va in cv.split(X_train,y_train):
        bundle=fit_model(name,X_train.iloc[tr],y_train.iloc[tr],w_train.iloc[tr])
        oof[va]=predict_probability(bundle,X_train.iloc[va])
    if np.isnan(oof).any(): raise RuntimeError("Missing OOF predictions.")
    return oof

def optimize_threshold(y_true,prob,weights):
    precision,recall,thresholds=precision_recall_curve(
        y_true,prob,sample_weight=weights)
    if len(thresholds)==0: return .5,np.nan
    f1=2*precision[:-1]*recall[:-1]/(precision[:-1]+recall[:-1]+1e-12)
    idx=int(np.nanargmax(f1))
    return float(thresholds[idx]),float(f1[idx])

oof_probabilities={}
threshold_rows=[]
for name in MODEL_PARAMS:
    oof=get_oof_probabilities(name)
    oof_probabilities[name]=oof
    threshold,oof_f1=optimize_threshold(y_train,oof,w_train)
    threshold_rows.append({"model":name,"optimized_threshold":threshold,
        "training_oof_weighted_f1":oof_f1,
        "training_oof_weighted_pr_auc":average_precision_score(y_train,oof,sample_weight=w_train)})
threshold_df=pd.DataFrame(threshold_rows)
threshold_map=dict(zip(threshold_df.model,threshold_df.optimized_threshold))
display(threshold_df)

## 9. Fit final models and evaluate on the held-out test set

In [ ]:
fitted_models,test_probabilities,test_predictions={}, {}, {}
metric_rows=[]
for name in MODEL_PARAMS:
    bundle=fit_model(name,X_train,y_train,w_train)
    fitted_models[name]=bundle
    prob=predict_probability(bundle,X_test)
    pred=(prob>=threshold_map[name]).astype(int)
    test_probabilities[name]=prob
    test_predictions[name]=pred
    metric_rows.append({
      "model":name,"threshold":threshold_map[name],"test_n":len(y_test),
      "weighted_roc_auc":roc_auc_score(y_test,prob,sample_weight=w_test),
      "weighted_pr_auc":average_precision_score(y_test,prob,sample_weight=w_test),
      "weighted_accuracy":accuracy_score(y_test,pred,sample_weight=w_test),
      "weighted_precision":precision_score(y_test,pred,sample_weight=w_test,zero_division=0),
      "weighted_recall":recall_score(y_test,pred,sample_weight=w_test,zero_division=0),
      "weighted_f1":f1_score(y_test,pred,sample_weight=w_test,zero_division=0),
      "unweighted_accuracy":accuracy_score(y_test,pred),
      "unweighted_precision":precision_score(y_test,pred,zero_division=0),
      "unweighted_recall":recall_score(y_test,pred,zero_division=0),
      "unweighted_f1":f1_score(y_test,pred,zero_division=0)})
results_df=pd.DataFrame(metric_rows).sort_values("weighted_pr_auc",ascending=False)
display(results_df)
for name in MODEL_PARAMS:
    print("\n",name,"threshold:",round(threshold_map[name],5))
    print(classification_report(y_test,test_predictions[name],
        target_names=["Not Any Wasting","Any Wasting"],zero_division=0))
    print("Confusion matrix:\n",confusion_matrix(y_test,test_predictions[name]))

## 10. ROC and precision-recall curves

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(14,5))
for name,prob in test_probabilities.items():
    RocCurveDisplay.from_predictions(y_test,prob,sample_weight=w_test,name=name,ax=axes[0])
    PrecisionRecallDisplay.from_predictions(y_test,prob,sample_weight=w_test,name=name,ax=axes[1])
axes[0].set_title("Survey-weighted ROC — held-out test")
axes[1].set_title("Survey-weighted precision-recall — held-out test")
plt.tight_layout()
plt.show()

## 11. Model-specific feature importance

In [ ]:
importance_frames=[]
for name,bundle in fitted_models.items():
    names=bundle["preprocessor"].get_feature_names_out()
    values=bundle["model"].feature_importances_
    frame=pd.DataFrame({"feature":names,"importance":values})
    frame["model"]=name
    importance_frames.append(frame)
importance_df=pd.concat(importance_frames,ignore_index=True)
for name in MODEL_PARAMS:
    top=importance_df[importance_df.model==name].nlargest(20,"importance").sort_values("importance")
    plt.figure(figsize=(9,7))
    plt.barh(top.feature,top.importance)
    plt.title(name+" — top 20 feature importances")
    plt.tight_layout()
    plt.show()

## 12. Save outputs and reproducibility manifest

In [ ]:
OUTPUT_DIR=Path("outputs")/"any_wasting_xgboost_lgbm_smote_weighted"
OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
cv_results.to_csv(OUTPUT_DIR/"cross_validation_results.csv",index=False)
threshold_df.to_csv(OUTPUT_DIR/"optimized_thresholds.csv",index=False)
results_df.to_csv(OUTPUT_DIR/"heldout_model_comparison.csv",index=False)
importance_df.to_csv(OUTPUT_DIR/"feature_importance.csv",index=False)
manifest={
 "target":TARGET,"predictors":PREDICTORS,"weight_column":WEIGHT_COL,
 "models":list(MODEL_PARAMS),"hyperparameter_tuning":"None; fixed configurations",
 "model_parameters":MODEL_PARAMS,"cv_folds":CV_FOLDS,
 "resampling":"SMOTENC applied within training folds only",
 "synthetic_sample_weight":"Mean original minority-class child sample weight in the corresponding training fold",
 "threshold_selection":"Maximize survey-weighted F1 using training-only out-of-fold probabilities",
 "thresholds":threshold_map,"cv_results":cv_results.to_dict(orient="records"),
 "heldout_metrics":results_df.to_dict(orient="records"),
 "warning":"Synthetic rows have approximate weights; SMOTENC does not preserve complex survey design."
}
with open(OUTPUT_DIR/"run_manifest.json","w",encoding="utf-8") as f:
    json.dump(manifest,f,indent=2)
print("Saved outputs to:",OUTPUT_DIR.resolve())

## 13. Methodological cautions

- CatBoost and Optuna are not used in this notebook.
- The two models use fixed hyperparameters; cross-validation estimates performance but does not tune the parameters.
- The child sample weight is a survey sampling weight, not a class-balancing weight. It is used in model fitting and weighted evaluation.
- SMOTENC synthetic rows have no genuine survey weights. Assigning the mean minority-class weight is an approximation.
- SMOTENC does not preserve MICS strata, clusters or other complex survey design features. This is a predictive workflow, not a replacement for survey-design-based inference.
- Thresholds maximize weighted F1 on training-only OOF predictions. They are data-dependent decision thresholds, not universal public-health cut-offs.
- Report weighted and unweighted metrics, and run a no-SMOTE sensitivity analysis.

In [ ]:
# Four-way comparison: SMOTENC/no child weights vs child weights/no SMOTENC
# The original notebook's hybrid model above is retained as a baseline.

STRATEGIES = {
    "SMOTENC_no_ch_weight": {"smote": True, "weights": False},
    "ch_weight_no_SMOTENC": {"smote": False, "weights": True},
}
COMPARISON_CONFIGS = [(m, s) for s in STRATEGIES for m in MODEL_PARAMS]

def fit_comparison_model(model_name, strategy, Xf, yf, wf):
    cfg = STRATEGIES[strategy]
    if cfg["smote"]:
        counts = pd.Series(yf).value_counts()
        if counts.min() < 2:
            raise ValueError("Not enough minority observations for SMOTENC.")
        sampler = SMOTENC(
            categorical_features=CAT_INDICES, random_state=SEED,
            k_neighbors=min(5, int(counts.min()) - 1)
        )
        Xfit, yfit = sampler.fit_resample(Xf, yf)
        if not isinstance(Xfit, pd.DataFrame):
            Xfit = pd.DataFrame(Xfit, columns=Xf.columns)
        for col in CATEGORICAL_COLUMNS:
            Xfit[col] = Xfit[col].astype(str)
        for col in NUMERIC_COLUMNS:
            Xfit[col] = pd.to_numeric(Xfit[col], errors="raise")
        fit_weights = None
    else:
        Xfit, yfit = Xf.copy(), np.asarray(yf, dtype=int)
        fit_weights = np.asarray(wf, dtype=float)
        fit_weights = fit_weights / fit_weights.mean()

    prep = make_preprocessor()
    Xfit_enc = prep.fit_transform(Xfit)
    if model_name == "XGBoost":
        estimator = XGBClassifier(
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=1, random_state=SEED,
            verbosity=0, **MODEL_PARAMS[model_name]
        )
    else:
        estimator = LGBMClassifier(
            objective="binary", n_jobs=1, random_state=SEED,
            verbosity=-1, **MODEL_PARAMS[model_name]
        )
    if fit_weights is None:
        estimator.fit(Xfit_enc, yfit)
    else:
        estimator.fit(Xfit_enc, yfit, sample_weight=fit_weights)
    return {"model": estimator, "preprocessor": prep}

def comparison_probability(bundle, Xnew):
    return bundle["model"].predict_proba(
        bundle["preprocessor"].transform(Xnew)
    )[:, 1]

# Cross-validation and out-of-fold probabilities; all resampling stays inside folds.
comparison_cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
comparison_oof = {}
comparison_thresholds = {}
comparison_cv_rows = []

for model_name, strategy in COMPARISON_CONFIGS:
    key = f"{model_name} | {strategy}"
    oof = np.full(len(X_train), np.nan)
    fold_scores = []
    use_weights = STRATEGIES[strategy]["weights"]

    for tr_idx, va_idx in comparison_cv.split(X_train, y_train):
        fitted = fit_comparison_model(
            model_name, strategy, X_train.iloc[tr_idx],
            y_train.iloc[tr_idx], w_train.iloc[tr_idx]
        )
        p = comparison_probability(fitted, X_train.iloc[va_idx])
        oof[va_idx] = p
        score_kwargs = {"sample_weight": w_train.iloc[va_idx]} if use_weights else {}
        fold_scores.append(average_precision_score(y_train.iloc[va_idx], p, **score_kwargs))

    if np.isnan(oof).any():
        raise RuntimeError(f"OOF predictions missing for {key}.")
    comparison_oof[key] = oof

    threshold_weights = w_train if use_weights else None
    precision, recall, thresholds = precision_recall_curve(
        y_train, oof, sample_weight=threshold_weights
    )
    f1_values = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-12
    )
    best_idx = int(np.nanargmax(f1_values))
    threshold = float(thresholds[best_idx]) if len(thresholds) else 0.5
    comparison_thresholds[key] = threshold

    comparison_cv_rows.append({
        "model": model_name, "strategy": strategy,
        "cv_pr_auc_type": "weighted" if use_weights else "unweighted",
        "mean_cv_pr_auc": float(np.mean(fold_scores)),
        "training_oof_f1": float(f1_values[best_idx]),
        "threshold": threshold
    })

comparison_cv_results = pd.DataFrame(comparison_cv_rows)
display(comparison_cv_results.round(4))

# Fit on the complete training split and evaluate every configuration on the same test split.
comparison_models = {}
comparison_predictions = {}
comparison_probabilities = {}
comparison_rows = []

for model_name, strategy in COMPARISON_CONFIGS:
    key = f"{model_name} | {strategy}"
    fitted = fit_comparison_model(model_name, strategy, X_train, y_train, w_train)
    probability = comparison_probability(fitted, X_test)
    prediction = (probability >= comparison_thresholds[key]).astype(int)
    comparison_models[key] = fitted
    comparison_probabilities[key] = probability
    comparison_predictions[key] = prediction

    comparison_rows.append({
        "model": model_name, "strategy": strategy,
        "threshold": comparison_thresholds[key],
        "weighted_precision": precision_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_recall": recall_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_f1": f1_score(y_test, prediction, sample_weight=w_test, zero_division=0),
        "weighted_pr_auc": average_precision_score(y_test, probability, sample_weight=w_test),
        "weighted_roc_auc": roc_auc_score(y_test, probability, sample_weight=w_test),
        "unweighted_precision": precision_score(y_test, prediction, zero_division=0),
        "unweighted_recall": recall_score(y_test, prediction, zero_division=0),
        "unweighted_f1": f1_score(y_test, prediction, zero_division=0),
        "unweighted_pr_auc": average_precision_score(y_test, probability),
        "unweighted_roc_auc": roc_auc_score(y_test, probability),
    })

comparison_results = pd.DataFrame(comparison_rows)
display(comparison_results.sort_values("weighted_f1", ascending=False).round(4))

# Compact precision/recall/F1 table, including both weighted and unweighted results.
prf_columns = [
    "model", "strategy", "weighted_precision", "weighted_recall", "weighted_f1",
    "unweighted_precision", "unweighted_recall", "unweighted_f1"
]
display(comparison_results[prf_columns].round(4))

# Generate an evidence-based discussion from this actual execution.
best_weighted_f1 = comparison_results.loc[comparison_results["weighted_f1"].idxmax()]
best_unweighted_f1 = comparison_results.loc[comparison_results["unweighted_f1"].idxmax()]
best_weighted_pr = comparison_results.loc[comparison_results["weighted_pr_auc"].idxmax()]
print("AUTOMATIC RESULTS DISCUSSION")
print("-" * 60)
print(
    f"Best weighted F1: {best_weighted_f1['model']} / {best_weighted_f1['strategy']} "
    f"({best_weighted_f1['weighted_f1']:.4f})."
)
print(
    f"Best unweighted F1: {best_unweighted_f1['model']} / {best_unweighted_f1['strategy']} "
    f"({best_unweighted_f1['unweighted_f1']:.4f})."
)
print(
    f"Best weighted PR-AUC: {best_weighted_pr['model']} / {best_weighted_pr['strategy']} "
    f"({best_weighted_pr['weighted_pr_auc']:.4f})."
)

smote_mean = comparison_results[
    comparison_results["strategy"] == "SMOTENC_no_ch_weight"
][["unweighted_precision", "unweighted_recall", "unweighted_f1", "unweighted_pr_auc"]].mean()
weight_mean = comparison_results[
    comparison_results["strategy"] == "ch_weight_no_SMOTENC"
][["unweighted_precision", "unweighted_recall", "unweighted_f1", "unweighted_pr_auc"]].mean()
print("\nMean unweighted metrics across XGBoost and LightGBM:")
print("SMOTENC/no child weights:", smote_mean.round(4).to_dict())
print("Child weights/no SMOTENC:", weight_mean.round(4).to_dict())

if smote_mean["unweighted_f1"] > weight_mean["unweighted_f1"]:
    print("In this run, SMOTENC/no child weights has the higher mean unweighted F1.")
elif smote_mean["unweighted_f1"] < weight_mean["unweighted_f1"]:
    print("In this run, child weights/no SMOTENC has the higher mean unweighted F1.")
else:
    print("In this run, the strategies have equal mean unweighted F1.")

print("\nInterpretation: higher recall means fewer wasting cases are missed, but may increase false positives.")
print("F1 balances precision and recall at the selected threshold; PR-AUC compares ranking across thresholds.")
print("Weighted metrics incorporate the child sample weights; unweighted metrics describe ordinary sample performance.")
print("Do not treat weighted and unweighted metrics as interchangeable or infer causal effects from this comparison.")

# Save comparison outputs separately from the original hybrid-baseline outputs.
COMPARISON_OUTPUT = Path("outputs") / "any_wasting_smotenc_vs_child_weights"
COMPARISON_OUTPUT.mkdir(parents=True, exist_ok=True)
comparison_cv_results.to_csv(COMPARISON_OUTPUT / "cross_validation.csv", index=False)
comparison_results.to_csv(COMPARISON_OUTPUT / "heldout_comparison.csv", index=False)
print("Comparison outputs saved to:", COMPARISON_OUTPUT.resolve())


### How to interpret the comparison

- **SMOTENC without child weights** asks how the classifiers perform after balancing the training data without incorporating survey sampling weights into model fitting.
- **Child weights without SMOTENC** asks how the classifiers perform on the original training observations when each observation contributes according to its child sampling weight.
- A higher recall can be valuable when missing a child with wasting is especially costly, but it may reduce precision by flagging more children who do not meet the outcome definition.
- Use weighted metrics when the intended interpretation concerns the population represented by the survey weights. Use unweighted metrics to describe the observed analytic sample.
- The strategy comparison is not a causal experiment: weighting and resampling alter the learning objective in different ways. SMOTENC also does not reproduce the survey's strata/cluster design.
- Do not report a winner until this notebook has been run against the revised MICS dataset.